# Aufgabe 4d: Citrus

Dieses Notebook verwendet die originale R-Implementierung `nolanlab/citrus` (Version 0.8) mit `Rclusterpp`, FlowCore und L1-regularisierter logistischer Regression aus `glmnet`.

Citrus führt hierarchisches Clustering ausschließlich mit den jeweiligen Trainingsspendern durch, berechnet spenderbezogene Clusterhäufigkeiten und bildet äußere Testspender anschließend in den bereits gelernten Clusterraum ab. Der Smoke-Modus verwendet wenige Zellen aus `gated_NK`; der finale Hauptvergleich verwendet `gated_alive`.

## 1. Separate R-Umgebung

Die Umgebung wird mit `environment-citrus.yml` erstellt. Da Citrus und Rclusterpp direkt aus den offiziellen GitHub-Repositories installiert werden, sind danach einmalig folgende Befehle nötig:

```r
remotes::install_github("nolanlab/Rclusterpp@a07380683ce7a6849af8ec27db6439ea3a707890", upgrade = "never", dependencies = FALSE)
remotes::install_github("nolanlab/citrus@d02baae544abdc403704aaceb75d1e7931a0331c", upgrade = "never", dependencies = FALSE)
IRkernel::installspec(name = "ssbi-citrus", displayname = "R (SSBI Citrus)")
```

In [1]:
required_packages <- c(
  "citrus", "Rclusterpp", "flowCore", "glmnet",
  "pamr", "samr", "igraph"
)
missing_packages <- required_packages[!vapply(
  required_packages, requireNamespace, logical(1), quietly = TRUE
)]
if (length(missing_packages) > 0) {
  stop("Fehlende Pakete: ", paste(missing_packages, collapse = ", "))
}
suppressPackageStartupMessages(library(citrus))
options(mc.cores = 1)
cat(
  "R:", R.version.string,
  "\nCitrus:", as.character(packageVersion("citrus")),
  "\nRclusterpp:", as.character(packageVersion("Rclusterpp")), "\n"
)

R: R version 4.5.3 (2026-03-11) 
Citrus: 0.8 
Rclusterpp: 0.2.6 


## 2. Konfiguration

`minimumClusterSizePercent = 0.05` ist in Citrus ein Anteil und entspricht 5 % der geclusterten Ereignisse. `cv.min` wählt die L1-Regularisierung mit minimalem innerem Klassifikationsfehler.

In [2]:
RUN_MODE <- Sys.getenv("TASK4_RUN_MODE", unset = "smoke")
RUN_ANALYSIS <- Sys.getenv("TASK4_RUN_TRAINING", unset = "1") == "1"
TRANSFORM_COFACTOR <- 5
MINIMUM_CLUSTER_SIZE_FRACTION <- 0.05
INNER_FOLD_COUNT <- 3L

if (RUN_MODE == "smoke") {
  GATE <- "gated_NK"
  SPLIT_IDS <- c(0L, 1L)
  FILE_SAMPLE_SIZE <- 100L
} else if (RUN_MODE == "full") {
  GATE <- "gated_alive"
  SPLIT_IDS <- 0:99
  FILE_SAMPLE_SIZE <- 1000L
} else {
  stop("Unbekannter RUN_MODE: ", RUN_MODE)
}
SPLIT_LIMIT <- as.integer(Sys.getenv("TASK4_SPLIT_LIMIT", unset = "0"))
if (SPLIT_LIMIT > 0L) {
  SPLIT_IDS <- head(SPLIT_IDS, SPLIT_LIMIT)
}
force_split_text <- Sys.getenv("TASK4_FORCE_SPLITS", unset = "")
FORCE_SPLIT_IDS <- if (nzchar(force_split_text)) {
  as.integer(strsplit(force_split_text, ",", fixed = TRUE)[[1]])
} else {
  integer()
}

gate_suffixes <- c(gated_NK = "_NK", gated_alive = "_alive")
GATE_SUFFIX <- unname(gate_suffixes[[GATE]])

working_directory <- normalizePath(getwd())
if (file.exists(file.path(working_directory, "AGENTS.md"))) {
  PROJECT_ROOT <- working_directory
} else {
  PROJECT_ROOT <- normalizePath(file.path(working_directory, ".."))
}
DATA_ROOT <- file.path(PROJECT_ROOT, "NK_cell_dataset", "NK_cell_dataset")
FCS_DIRECTORY <- file.path(DATA_ROOT, "NK_cell_dataset", GATE)
MARKERS_PATH <- file.path(DATA_ROOT, "NK_markers.csv")
SPLITS_PATH <- file.path(PROJECT_ROOT, "results", "tables", "task4_donor_splits.csv")
PREDICTIONS_PATH <- file.path(
  PROJECT_ROOT, "results", "tables",
  paste0("task4_citrus_predictions_", GATE, "_", RUN_MODE, ".csv")
)
SELECTION_PATH <- file.path(
  PROJECT_ROOT, "results", "tables",
  paste0("task4_citrus_selection_", GATE, "_", RUN_MODE, ".csv")
)
CLUSTERS_PATH <- file.path(
  PROJECT_ROOT, "results", "tables",
  paste0("task4_citrus_clusters_", GATE, "_", RUN_MODE, ".csv")
)

required_paths <- c(FCS_DIRECTORY, MARKERS_PATH, SPLITS_PATH)
if (!all(file.exists(required_paths))) {
  stop("Mindestens ein erwarteter Datenpfad fehlt.")
}
cat(
  "Modus:", RUN_MODE, "| Gate:", GATE,
  "| Zellen je Datei:", FILE_SAMPLE_SIZE,
  "| Training:", if (RUN_ANALYSIS) "aktiv" else "deaktiviert", "\n"
)

Modus: smoke | Gate: gated_NK | Zellen je Datei: 100 | Training: deaktiviert 


## 3. Marker und gemeinsame Splits

Die in Python erzeugten unsigned 32-Bit-Seeds können größer als Rs signed Integermaximum sein. Für R wird deshalb reproduzierbar `split_seed modulo .Machine$integer.max` verwendet. Die Spenderzuordnungen selbst bleiben unverändert.

In [3]:
markers <- scan(MARKERS_PATH, what = character(), sep = ",", quiet = TRUE)
donor_splits <- read.csv(SPLITS_PATH, stringsAsFactors = FALSE)

stopifnot(length(markers) == 37L)
stopifnot(length(unique(markers)) == 37L)
stopifnot(all(SPLIT_IDS %in% donor_splits$split_id))
stopifnot(!anyDuplicated(donor_splits[c("split_id", "donor_id")]))
stopifnot(all(table(
  donor_splits$split_id, donor_splits$outer_partition
)[, c("test", "train")] == matrix(c(6L, 14L), nrow = 100L, ncol = 2L, byrow = TRUE)))

head(donor_splits)

,split_id,split_seed,donor_id,label,outer_partition,inner_fold
,<int>,<dbl>,<chr>,<int>,<chr>,<int>
1,0,3003105692,a_001,1,train,0
2,0,3003105692,a_002,1,test,-1
3,0,3003105692,a_003,0,train,0
4,0,3003105692,a_004,0,test,-1
5,0,3003105692,a_005,1,train,1
6,0,3003105692,a_006,0,test,-1


## 4. Offizielles Citrus mit vorgegebenen inneren Folds

Die öffentliche Funktion `citrus.clusterAndMapFolds` erzeugt eigene zufällige Folds und akzeptiert keine vorgegebene Zuordnung. Damit Citrus exakt dieselben inneren Spenderfolds wie SVM und CellCNN verwendet, wird hier das entsprechende `citrus.foldClustering`-Objekt aus den offiziellen Citrus-Funktionen `citrus.clusterFold`, `citrus.mapFoldDataToClusterSpace` und `citrus.cluster` aufgebaut. An Clustering, Featureberechnung und glmnet-Modell wird nichts ersetzt.

In [4]:
read_citrus_files <- function(sample_table, seed) {
  set.seed(seed)
  file_list <- data.frame(
    unstim = paste0(sample_table$donor_id, GATE_SUFFIX, ".fcs"),
    stringsAsFactors = FALSE
  )
  citrus.readFCSSet(
    dataDirectory = FCS_DIRECTORY,
    fileList = file_list,
    fileSampleSize = FILE_SAMPLE_SIZE,
    transformColumns = markers,
    transformCofactor = TRANSFORM_COFACTOR,
    useChannelDescriptions = TRUE
  )
}

build_fixed_fold_clustering <- function(train_fcs, train_table) {
  folds <- lapply(0:(INNER_FOLD_COUNT - 1L), function(fold) {
    which(train_table$inner_fold == fold)
  })
  stopifnot(length(unique(unlist(folds))) == nrow(train_table))
  stopifnot(length(intersect(folds[[1]], folds[[2]])) == 0L)
  stopifnot(length(intersect(folds[[1]], folds[[3]])) == 0L)
  stopifnot(length(intersect(folds[[2]], folds[[3]])) == 0L)

  result <- list()
  result$folds <- folds
  result$foldClustering <- lapply(
    seq_along(folds),
    citrus:::citrus.clusterFold,
    folds = folds,
    citrus.combinedFCSSet = train_fcs,
    clusteringColumns = markers
  )
  result$foldMappingAssignments <- lapply(
    seq_along(folds),
    citrus:::citrus.mapFoldDataToClusterSpace,
    folds = folds,
    foldClustering = result$foldClustering,
    citrus.combinedFCSSet = train_fcs
  )
  result$allClustering <- citrus.cluster(train_fcs, markers)
  result$nFolds <- length(folds)
  class(result) <- "citrus.foldClustering"
  result
}

extract_selected_cluster_profiles <- function(
  split_id, split_seed, train_fcs, fold_clustering, fold_features, regression
) {
  lambda <- regression$cvMinima[["cv.min"]]
  coefficients <- as.matrix(predict(
    regression$finalModel$model,
    newx = fold_features$allFeatures,
    type = "coefficients",
    s = lambda
  ))
  differential <- regression$differentialFeatures[["cv.min"]]
  selected_cluster_ids <- differential$clusters
  if (is.null(selected_cluster_ids) || length(selected_cluster_ids) == 0L) {
    return(empty_cluster_table)
  }

  records <- list()
  record_index <- 1L
  for (cluster_id in selected_cluster_ids) {
    event_indices <- fold_clustering$allClustering$clusterMembership[[cluster_id]]
    centroid <- colMeans(train_fcs$data[event_indices, markers, drop = FALSE])
    feature_name <- paste("cluster", cluster_id, "abundance")
    coefficient <- if (feature_name %in% rownames(coefficients)) {
      as.numeric(coefficients[feature_name, 1])
    } else {
      NA_real_
    }
    for (marker in markers) {
      records[[record_index]] <- data.frame(
        method = "citrus", gate = GATE, run_mode = RUN_MODE,
        split_id = split_id, split_seed = split_seed,
        cluster_id = cluster_id, marker = marker,
        centroid = centroid[[marker]], coefficient = coefficient,
        transform_cofactor = TRANSFORM_COFACTOR,
        minimum_cluster_size_fraction = MINIMUM_CLUSTER_SIZE_FRACTION
      )
      record_index <- record_index + 1L
    }
  }
  do.call(rbind, records)
}

run_citrus_split <- function(split_id) {
  split <- donor_splits[donor_splits$split_id == split_id, ]
  train <- split[split$outer_partition == "train", ]
  test <- split[split$outer_partition == "test", ]
  train <- train[order(train$donor_id), ]
  test <- test[order(test$donor_id), ]
  stopifnot(nrow(train) == 14L, nrow(test) == 6L)
  stopifnot(length(intersect(train$donor_id, test$donor_id)) == 0L)

  split_seed <- unique(split$split_seed)
  stopifnot(length(split_seed) == 1L)
  r_seed <- as.integer(split_seed %% .Machine$integer.max)
  train_labels <- factor(
    ifelse(train$label == 1L, "CMV+", "CMV-"),
    levels = c("CMV-", "CMV+")
  )

  train_fcs <- read_citrus_files(train, r_seed)
  fold_clustering <- build_fixed_fold_clustering(train_fcs, train)
  fold_features <- citrus.calculateFoldFeatureSet(
    citrus.foldClustering = fold_clustering,
    citrus.combinedFCSSet = train_fcs,
    featureType = "abundances",
    minimumClusterSizePercent = MINIMUM_CLUSTER_SIZE_FRACTION
  )
  regression <- citrus.endpointRegress(
    modelType = "glmnet",
    citrus.foldFeatureSet = fold_features,
    labels = train_labels,
    family = "classification"
  )

  test_fcs <- read_citrus_files(test, r_seed + 1L)
  test_mapping <- citrus.mapToClusterSpace(
    citrus.combinedFCSSet.new = test_fcs,
    citrus.combinedFCSSet.old = train_fcs,
    citrus.clustering = fold_clustering$allClustering,
    mappingColumns = markers,
    mc.cores = 1
  )
  test_features <- citrus.calculateFeatures(
    citrus.combinedFCSSet = test_fcs,
    clusterAssignments = test_mapping$clusterMembership,
    clusterIds = fold_features$allLargeEnoughClusters,
    featureType = "abundances"
  )
  lambda <- regression$cvMinima[["cv.min"]]
  scores <- as.numeric(predict(
    regression$finalModel$model,
    newx = test_features,
    type = "response",
    s = lambda
  ))
  scores <- round(scores, digits = 15L)
  stopifnot(length(scores) == nrow(test))
  stopifnot(all(is.finite(scores)), all(scores >= 0 & scores <= 1))

  selected_clusters <- regression$differentialFeatures[["cv.min"]]$clusters
  selected_cluster_count <- if (is.null(selected_clusters)) 0L else length(selected_clusters)
  predictions <- data.frame(
    method = "citrus", gate = GATE, run_mode = RUN_MODE,
    split_id = split_id, split_seed = split_seed, donor_id = test$donor_id,
    y_true = test$label, score = scores, decision_threshold = 0.5,
    y_pred = as.integer(scores >= 0.5), file_sample_size = FILE_SAMPLE_SIZE,
    minimum_cluster_size_fraction = MINIMUM_CLUSTER_SIZE_FRACTION,
    selected_lambda = lambda, selected_cluster_count = selected_cluster_count
  )
  selection <- data.frame(
    method = "citrus", gate = GATE, run_mode = RUN_MODE,
    split_id = split_id, split_seed = split_seed, r_seed = r_seed,
    selected_lambda = lambda, selected_cluster_count = selected_cluster_count,
    eligible_cluster_count = length(fold_features$allLargeEnoughClusters),
    file_sample_size = FILE_SAMPLE_SIZE,
    minimum_cluster_size_fraction = MINIMUM_CLUSTER_SIZE_FRACTION
  )
  cluster_profiles <- extract_selected_cluster_profiles(
    split_id, split_seed, train_fcs, fold_clustering, fold_features, regression
  )
  list(predictions = predictions, selection = selection, clusters = cluster_profiles)
}

## 5. Citrus ausführen oder vorhandene Ergebnisse laden

Die Warnung von glmnet, dass eine Klasse weniger als acht Beobachtungen enthält, ist bei exakt sieben Trainingsspendern je Klasse unvermeidbar und unterstreicht die kleine Kohorte.

In [5]:
empty_cluster_table <- data.frame(
  method = character(), gate = character(), run_mode = character(),
  split_id = integer(), split_seed = numeric(), cluster_id = integer(),
  marker = character(), centroid = numeric(), coefficient = numeric(),
  transform_cofactor = numeric(),
  minimum_cluster_size_fraction = numeric()
)

citrus_predictions <- data.frame()
citrus_selection <- data.frame()
citrus_clusters <- empty_cluster_table
completed_split_ids <- integer()
if (
  RUN_ANALYSIS && file.exists(PREDICTIONS_PATH) &&
  file.exists(SELECTION_PATH) && file.exists(CLUSTERS_PATH)
) {
  previous_predictions <- read.csv(PREDICTIONS_PATH, stringsAsFactors = FALSE)
  previous_selection <- read.csv(SELECTION_PATH, stringsAsFactors = FALSE)
  previous_clusters <- read.csv(CLUSTERS_PATH, stringsAsFactors = FALSE)
  reusable <- (
    all(previous_predictions$gate == GATE) &&
    all(previous_predictions$run_mode == RUN_MODE) &&
    all(previous_predictions$file_sample_size == FILE_SAMPLE_SIZE) &&
    all(
      previous_predictions$minimum_cluster_size_fraction ==
        MINIMUM_CLUSTER_SIZE_FRACTION
    )
  )
  if (reusable) {
    citrus_predictions <- previous_predictions[
      !previous_predictions$split_id %in% FORCE_SPLIT_IDS,
    ]
    citrus_selection <- previous_selection[
      !previous_selection$split_id %in% FORCE_SPLIT_IDS,
    ]
    citrus_clusters <- previous_clusters[
      !previous_clusters$split_id %in% FORCE_SPLIT_IDS,
    ]
    completed_split_ids <- unique(citrus_predictions$split_id)
    cat("Vorhandene Citrus-Ergebnisse geladen:", length(completed_split_ids), "Splits.\n")
  }
}

if (RUN_ANALYSIS) {
  for (split_id in SPLIT_IDS) {
    if (split_id %in% completed_split_ids) next
    result <- run_citrus_split(split_id)
    citrus_predictions <- rbind(citrus_predictions, result$predictions)
    citrus_selection <- rbind(citrus_selection, result$selection)
    citrus_clusters <- rbind(citrus_clusters, result$clusters)
    write.csv(citrus_predictions, PREDICTIONS_PATH, row.names = FALSE)
    write.csv(citrus_selection, SELECTION_PATH, row.names = FALSE)
    write.csv(citrus_clusters, CLUSTERS_PATH, row.names = FALSE)
    cat("Split", split_id, "abgeschlossen.\n")
  }
} else {
  cat("Training deaktiviert; vorhandene Citrus-Ergebnisse werden geladen.\n")
  citrus_predictions <- read.csv(PREDICTIONS_PATH, stringsAsFactors = FALSE)
  citrus_selection <- read.csv(SELECTION_PATH, stringsAsFactors = FALSE)
  citrus_clusters <- read.csv(CLUSTERS_PATH, stringsAsFactors = FALSE)
}

stopifnot(all(sort(unique(citrus_predictions$split_id)) == sort(SPLIT_IDS)))
stopifnot(all(citrus_predictions$gate == GATE))
stopifnot(all(citrus_predictions$run_mode == RUN_MODE))
stopifnot(all(citrus_predictions$split_seed == donor_splits$split_seed[
  match(
    paste(citrus_predictions$split_id, citrus_predictions$donor_id),
    paste(donor_splits$split_id, donor_splits$donor_id)
  )
]))
stopifnot(all(citrus_predictions$file_sample_size == FILE_SAMPLE_SIZE))
stopifnot(all(
  citrus_predictions$minimum_cluster_size_fraction == MINIMUM_CLUSTER_SIZE_FRACTION
))
stopifnot(all(citrus_predictions$decision_threshold == 0.5))
stopifnot(all(table(citrus_predictions$split_id) == 6L))
stopifnot(!anyDuplicated(citrus_predictions[c("split_id", "donor_id")]))
stopifnot(all(is.finite(citrus_predictions$score)))
stopifnot(all(citrus_predictions$score >= 0 & citrus_predictions$score <= 1))
stopifnot(nrow(citrus_selection) == length(SPLIT_IDS))
stopifnot(!anyDuplicated(citrus_selection$split_id))
stopifnot(all(sort(citrus_selection$split_id) == sort(SPLIT_IDS)))
stopifnot(all(citrus_selection$gate == GATE))
stopifnot(all(citrus_selection$run_mode == RUN_MODE))
stopifnot(all(citrus_selection$file_sample_size == FILE_SAMPLE_SIZE))
stopifnot(all(
  citrus_selection$minimum_cluster_size_fraction == MINIMUM_CLUSTER_SIZE_FRACTION
))
expected_split_seeds <- unique(donor_splits[
  donor_splits$split_id %in% SPLIT_IDS, c("split_id", "split_seed")
])
stopifnot(nrow(merge(
  citrus_selection[c("split_id", "split_seed")], expected_split_seeds,
  by = c("split_id", "split_seed")
)) == length(SPLIT_IDS))
if (nrow(citrus_clusters) > 0L) {
  stopifnot(all(citrus_clusters$gate == GATE))
  stopifnot(all(citrus_clusters$run_mode == RUN_MODE))
  stopifnot(all(citrus_clusters$transform_cofactor == TRANSFORM_COFACTOR))
  stopifnot(all(
    citrus_clusters$minimum_cluster_size_fraction == MINIMUM_CLUSTER_SIZE_FRACTION
  ))
  stopifnot(all(is.finite(citrus_clusters$centroid)))
  stopifnot(all(is.finite(citrus_clusters$coefficient)))
  profile_sizes <- aggregate(
    marker ~ split_id + cluster_id, citrus_clusters, length
  )
  stopifnot(all(profile_sizes$marker == 37L))
  observed_cluster_counts <- aggregate(
    cluster_id ~ split_id, unique(citrus_clusters[c("split_id", "cluster_id")]), length
  )
  names(observed_cluster_counts)[2] <- "selected_cluster_count"
  expected_nonempty <- citrus_selection[
    citrus_selection$selected_cluster_count > 0L,
    c("split_id", "selected_cluster_count")
  ]
  stopifnot(nrow(merge(
    observed_cluster_counts, expected_nonempty,
    by = c("split_id", "selected_cluster_count")
  )) == nrow(expected_nonempty))
} else {
  stopifnot(all(citrus_selection$selected_cluster_count == 0L))
}

expected_test <- donor_splits[
  donor_splits$split_id %in% SPLIT_IDS & donor_splits$outer_partition == "test",
  c("split_id", "donor_id", "label")
]
names(expected_test)[names(expected_test) == "label"] <- "y_true"
matched <- merge(
  citrus_predictions[c("split_id", "donor_id", "y_true")],
  expected_test, by = c("split_id", "donor_id", "y_true")
)
stopifnot(nrow(matched) == nrow(expected_test), nrow(matched) == nrow(citrus_predictions))

citrus_selection
citrus_predictions

Training deaktiviert; vorhandene Citrus-Ergebnisse werden geladen.


method,gate,run_mode,split_id,split_seed,r_seed,selected_lambda,selected_cluster_count,eligible_cluster_count,file_sample_size,minimum_cluster_size_fraction
<chr>,<chr>,<chr>,<int>,<dbl>,<int>,<dbl>,<int>,<int>,<int>,<dbl>
citrus,gated_NK,smoke,0,3003105692,855622045,0.26526904,0,31,100,0.05
citrus,gated_NK,smoke,1,976400780,976400780,0.01741989,12,33,100,0.05


method,gate,run_mode,split_id,split_seed,donor_id,y_true,score,decision_threshold,y_pred,file_sample_size,minimum_cluster_size_fraction,selected_lambda,selected_cluster_count
<chr>,<chr>,<chr>,<int>,<dbl>,<chr>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<int>
citrus,gated_NK,smoke,0,3003105692,a_002,1,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,0,3003105692,a_004,0,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,0,3003105692,a_006,0,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,0,3003105692,a_1a,0,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,0,3003105692,a_2a,0,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,0,3003105692,a_4a,1,0.500000000,0.5,1,100,0.05,0.26526904,0
citrus,gated_NK,smoke,1,976400780,a_002,1,0.747740183,0.5,1,100,0.05,0.01741989,12
citrus,gated_NK,smoke,1,976400780,a_003,0,0.788282066,0.5,1,100,0.05,0.01741989,12
citrus,gated_NK,smoke,1,976400780,a_004,0,0.004411015,0.5,0,100,0.05,0.01741989,12


## 6. Spenderbezogene Smoke-Test-Metriken

Diese lokale Berechnung dient nur der isolierten technischen Prüfung. Der finale Vergleich berechnet alle Methodenmetriken einheitlich im Python-Vergleichsnotebook.

In [6]:
rank_roc_auc <- function(y_true, scores) {
  positive_count <- sum(y_true == 1L)
  negative_count <- sum(y_true == 0L)
  ranks <- rank(scores, ties.method = "average")
  (sum(ranks[y_true == 1L]) - positive_count * (positive_count + 1) / 2) /
    (positive_count * negative_count)
}
balanced_accuracy <- function(y_true, y_pred) {
  sensitivity <- mean(y_pred[y_true == 1L] == 1L)
  specificity <- mean(y_pred[y_true == 0L] == 0L)
  (sensitivity + specificity) / 2
}

metric_records <- lapply(split(citrus_predictions, citrus_predictions$split_id), function(x) {
  data.frame(
    split_id = unique(x$split_id),
    roc_auc = rank_roc_auc(x$y_true, x$score),
    balanced_accuracy = balanced_accuracy(x$y_true, x$y_pred)
  )
})
citrus_metrics <- do.call(rbind, metric_records)
rownames(citrus_metrics) <- NULL
citrus_metrics

split_id,roc_auc,balanced_accuracy
<int>,<dbl>,<dbl>
0,0.50,0.500
1,0.75,0.625


## Abschluss von Schritt 5

Die originale Citrus-R-Pipeline ist in die gemeinsamen äußeren und inneren Spendersplits integriert. Clustering, Auswahl zulässiger Cluster, Abundanzfeatures und L1-Merkmalsauswahl verwenden ausschließlich Trainingsspender; Testspender werden nur in den fertigen Clusterraum abgebildet. Smoke-Ergebnisse sind nicht für den Bericht bestimmt.